Total crime count (Outputs Crime Count input - Location and Date)
- Ridge & Lasso Regression
- Random Forest Regressor
- XGBoost Regressor
- SARIMA 
- LSTM


# Model 01 - Ridge & Lasso Regression


In [1]:
# =============================================================================
# Model 1 — Ridge & Lasso Regression
# Task    : Predict Crime_Count based on Location and Date
# =============================================================================

import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore")


In [2]:
# =============================================================================
# 1. LOAD DATA
# =============================================================================
# Replace this path with your actual file path
df = pd.read_csv("./crime_per_capita_df_corrected.csv")

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['Year'].min()}-{df['Month'].min():02d} to "
      f"{df['Year'].max()}-{df['Month'].max():02d}")
print(f"Unique LSOAs: {df['LSOA_Code'].nunique()}")


Dataset shape: (304336, 39)
Date range: 2020-01 to 2024-12
Unique LSOAs: 12415


In [3]:
# =============================================================================
# 2. FEATURE SELECTION
# =============================================================================
# Features used: location + date + engineered temporal/socioeconomic columns
FEATURES = [
    # --- Location ---
    "LSOA_Latitude",
    "LSOA_Longitude",
    "LSOA_Shape_Area",
    "LSOA_Shape_Length",

    # --- Date ---
    "Year",
    "Month",
    "month_sin",
    "month_cos",

    # --- Season flags ---
    "is_summer",
    "is_winter",
    "is_spring",
    "is_autumn",

    # --- Socioeconomic ---
    "Total_Population",
    "pop_density",
    "income_per_capita",
    "Income_Deprivation",

    # --- Weather ---
    "Max_Temperature_Celsius",
    "Min_Temperature_Celsius",
    "Rainfall_mm",
    "Sunshine_Hours",
    "Air_Frost_Days",

    # --- Lag & rolling features ---
    "crime_lag_1m",
    "crime_lag_2m",
    "crime_lag_3m",
    "crime_lag_6m",
    "crime_rolling_3m_mean",
    "crime_rolling_3m_std",
    "crime_rolling_6m_mean",
    "crime_rolling_6m_std",
    "msoa_avg_crime",
]

TARGET = "Crime_Count"

# Verify all features exist in the dataframe
missing = [f for f in FEATURES if f not in df.columns]
if missing:
    print(f"\nWARNING — missing columns: {missing}")
    FEATURES = [f for f in FEATURES if f in df.columns]

print(f"\nUsing {len(FEATURES)} features to predict '{TARGET}'")


Using 30 features to predict 'Crime_Count'


In [4]:
# =============================================================================
# 3. TRAIN / TEST SPLIT  (time-based — never random for temporal data)
# =============================================================================
# Train: 2020–2023  |  Test: 2024
TRAIN_YEARS = [2020, 2021, 2022, 2023]
TEST_YEARS  = [2024]

train = df[df["Year"].isin(TRAIN_YEARS)].copy()
test  = df[df["Year"].isin(TEST_YEARS)].copy()

X_train = train[FEATURES]
y_train = train[TARGET]
X_test  = test[FEATURES]
y_test  = test[TARGET]

print(f"\nTrain size : {len(X_train):,} rows ({TRAIN_YEARS[0]}–{TRAIN_YEARS[-1]})")
print(f"Test size  : {len(X_test):,} rows  ({TEST_YEARS[0]})")



Train size : 241,710 rows (2020–2023)
Test size  : 62,626 rows  (2024)


In [5]:
# =============================================================================
# 4. SCALING
# Linear models are sensitive to feature scale — StandardScaler is required
# =============================================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)       # use train stats on test

In [6]:
# =============================================================================
# 5. HELPER — evaluation metrics
# =============================================================================
def evaluate(model_name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100

    print(f"\n{'='*45}")
    print(f"  {model_name}")
    print(f"{'='*45}")
    print(f"  MAE   : {mae:.4f}   (avg error in crime count units)")
    print(f"  RMSE  : {rmse:.4f}  (penalises large errors more)")
    print(f"  R²    : {r2:.4f}   (1.0 = perfect, 0 = baseline mean)")
    print(f"  MAPE  : {mape:.2f}%  (mean absolute % error)")

    return {"Model": model_name, "MAE": round(mae,4), "RMSE": round(rmse,4),
            "R2": round(r2,4), "MAPE": round(mape,2)}

In [7]:
# =============================================================================
# 6. LASSO REGRESSION
# Lasso can zero out weak features (useful for feature selection insight)
# =============================================================================
print("\n--- Fitting Lasso Regression ---")

lasso_alphas = [0.01, 0.1, 1.0, 10.0]
lasso_results = []

for alpha in lasso_alphas:
    lasso = Lasso(alpha=alpha, max_iter=5000)
    lasso.fit(X_train_scaled, y_train)
    preds = lasso.predict(X_test_scaled)
    preds = np.clip(preds, 0, None)
    res = evaluate(f"Lasso (alpha={alpha})", y_test, preds)

    # Count how many features Lasso zeroed out
    n_zero = np.sum(lasso.coef_ == 0)
    print(f"  Features zeroed out: {n_zero} / {len(FEATURES)}")

    lasso_results.append({**res, "alpha": alpha, "model_obj": lasso})

best_lasso = min(lasso_results, key=lambda x: x["MAE"])
print(f"\nBest Lasso alpha: {best_lasso['alpha']}  (MAE={best_lasso['MAE']})")


--- Fitting Lasso Regression ---

  Lasso (alpha=0.01)
  MAE   : 4.8124   (avg error in crime count units)
  RMSE  : 8.9729  (penalises large errors more)
  R²    : 0.9382   (1.0 = perfect, 0 = baseline mean)
  MAPE  : 44.25%  (mean absolute % error)
  Features zeroed out: 0 / 30

  Lasso (alpha=0.1)
  MAE   : 4.6965   (avg error in crime count units)
  RMSE  : 8.7972  (penalises large errors more)
  R²    : 0.9406   (1.0 = perfect, 0 = baseline mean)
  MAPE  : 43.17%  (mean absolute % error)
  Features zeroed out: 15 / 30

  Lasso (alpha=1.0)
  MAE   : 4.6464   (avg error in crime count units)
  RMSE  : 8.7064  (penalises large errors more)
  R²    : 0.9418   (1.0 = perfect, 0 = baseline mean)
  MAPE  : 47.44%  (mean absolute % error)
  Features zeroed out: 24 / 30

  Lasso (alpha=10.0)
  MAE   : 7.0165   (avg error in crime count units)
  RMSE  : 17.7248  (penalises large errors more)
  R²    : 0.7588   (1.0 = perfect, 0 = baseline mean)
  MAPE  : 103.73%  (mean absolute % error)
  

In [8]:
# =============================================================================
# 7. RIDGE REGRESSION
# alpha = regularisation strength (higher = more regularisation)
# We try a few alpha values so you can see the effect
# =============================================================================
print("\n--- Fitting Ridge Regression ---")

ridge_alphas = [0.1, 1.0, 10.0, 100.0]
ridge_results = []

for alpha in ridge_alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)
    preds = ridge.predict(X_test_scaled)
    preds = np.clip(preds, 0, None)           # crime count can't be negative
    res = evaluate(f"Ridge (alpha={alpha})", y_test, preds)
    ridge_results.append({**res, "alpha": alpha, "model_obj": ridge})

# Pick the best Ridge by MAE
best_ridge = min(ridge_results, key=lambda x: x["MAE"])
print(f"\nBest Ridge alpha: {best_ridge['alpha']}  (MAE={best_ridge['MAE']})")


--- Fitting Ridge Regression ---

  Ridge (alpha=0.1)
  MAE   : 4.8172   (avg error in crime count units)
  RMSE  : 8.9934  (penalises large errors more)
  R²    : 0.9379   (1.0 = perfect, 0 = baseline mean)
  MAPE  : 44.44%  (mean absolute % error)

  Ridge (alpha=1.0)
  MAE   : 4.8172   (avg error in crime count units)
  RMSE  : 8.9934  (penalises large errors more)
  R²    : 0.9379   (1.0 = perfect, 0 = baseline mean)
  MAPE  : 44.44%  (mean absolute % error)

  Ridge (alpha=10.0)
  MAE   : 4.8173   (avg error in crime count units)
  RMSE  : 8.9932  (penalises large errors more)
  R²    : 0.9379   (1.0 = perfect, 0 = baseline mean)
  MAPE  : 44.44%  (mean absolute % error)

  Ridge (alpha=100.0)
  MAE   : 4.8180   (avg error in crime count units)
  RMSE  : 8.9916  (penalises large errors more)
  R²    : 0.9379   (1.0 = perfect, 0 = baseline mean)
  MAPE  : 44.43%  (mean absolute % error)

Best Ridge alpha: 0.1  (MAE=4.8172)


In [9]:
# =============================================================================
# 8. FEATURE IMPORTANCE via Ridge coefficients
# (absolute coefficient value after scaling = importance)
# =============================================================================
print("\n--- Top 10 Most Important Features (Best Ridge) ---")
best_ridge_model = best_ridge["model_obj"]
coef_df = pd.DataFrame({
    "Feature"    : FEATURES,
    "Coefficient": best_ridge_model.coef_,
    "Abs_Coef"   : np.abs(best_ridge_model.coef_)
}).sort_values("Abs_Coef", ascending=False)

print(coef_df[["Feature", "Coefficient"]].head(10).to_string(index=False))



--- Top 10 Most Important Features (Best Ridge) ---
                Feature  Coefficient
           crime_lag_1m    10.336031
  crime_rolling_3m_mean     9.101642
Max_Temperature_Celsius    -8.699768
Min_Temperature_Celsius     8.498290
         Sunshine_Hours     3.697939
           crime_lag_6m     3.367205
              month_cos     2.836787
           crime_lag_2m     1.906111
           crime_lag_3m    -1.879000
                  Month    -1.863442


In [10]:
# =============================================================================
# 9. LASSO FEATURE SELECTION — which features survived?
# =============================================================================
print("\n--- Features Kept by Best Lasso (non-zero coefficients) ---")
best_lasso_model = best_lasso["model_obj"]
lasso_coef_df = pd.DataFrame({
    "Feature"    : FEATURES,
    "Coefficient": best_lasso_model.coef_
})
kept = lasso_coef_df[lasso_coef_df["Coefficient"] != 0].sort_values(
    "Coefficient", key=abs, ascending=False
)
dropped = lasso_coef_df[lasso_coef_df["Coefficient"] == 0]
print(f"Kept   : {len(kept)} features")
print(f"Dropped: {len(dropped)} features")
print(kept[["Feature", "Coefficient"]].to_string(index=False))


--- Features Kept by Best Lasso (non-zero coefficients) ---
Kept   : 6 features
Dropped: 24 features
              Feature  Coefficient
         crime_lag_1m    12.242361
crime_rolling_6m_mean     4.178111
crime_rolling_3m_mean     2.872503
         crime_lag_2m     2.053015
         crime_lag_6m     0.969571
 crime_rolling_6m_std     0.049654


In [11]:
# =============================================================================
# 10. COMPARISON TABLE
# =============================================================================
print("\n\n--- FINAL COMPARISON: Best Ridge vs Best Lasso ---")
summary = pd.DataFrame([
    {k: v for k, v in best_ridge.items() if k != "model_obj"},
    {k: v for k, v in best_lasso.items() if k != "model_obj"},
])
print(summary[["Model", "MAE", "RMSE", "R2", "MAPE"]].to_string(index=False))



--- FINAL COMPARISON: Best Ridge vs Best Lasso ---
            Model    MAE   RMSE     R2  MAPE
Ridge (alpha=0.1) 4.8172 8.9934 0.9379 44.44
Lasso (alpha=1.0) 4.6464 8.7064 0.9418 47.44


In [12]:
# =============================================================================
# 11. SAVE RESULTS TO CSV  (for comparison with future models)
# =============================================================================
summary[["Model", "MAE", "RMSE", "R2", "MAPE"]].to_csv(
    "model_comparison.csv", index=False
)
print("\nResults saved to model_comparison.csv")
print("(You will append other models to this file as you build them)")



Results saved to model_comparison.csv
(You will append other models to this file as you build them)


In [13]:
# =============================================================================
# 12. SAMPLE PREDICTIONS  - Lasso Model
# =============================================================================
print("\n--- Sample Predictions vs Actual (first 10 test rows) ---")
best_lasso_preds = np.clip(best_lasso_model.predict(X_test_scaled), 0, None)
sample = test[["LSOA_Code", "Year", "Month", TARGET]].copy().head(10)
sample["Predicted"] = np.round(best_lasso_preds[:10], 1)
sample["Error"]     = np.round(np.abs(sample[TARGET] - sample["Predicted"]), 1)
print(sample.to_string(index=False))


--- Sample Predictions vs Actual (first 10 test rows) ---
LSOA_Code  Year  Month  Crime_Count  Predicted  Error
E01000001  2024      1            2        2.6    0.6
E01000001  2024      2            1        3.0    2.0
E01000001  2024      3            2        2.5    0.5
E01000001  2024      4            1        3.0    2.0
E01000001  2024      5            2        2.5    0.5
E01000001  2024      7            1        2.9    1.9
E01000001  2024      8            2        2.5    0.5
E01000001  2024      9            1        2.9    1.9
E01000001  2024     10            4        2.5    1.5
E01000001  2024     11            5        4.1    0.9


In [14]:
# =============================================================================
# 12. SAMPLE PREDICTIONS  - Ridge Model
# =============================================================================
print("\n--- Sample Predictions vs Actual (first 10 test rows) ---")
best_ridge_preds = np.clip(best_ridge_model.predict(X_test_scaled), 0, None)
sample = test[["LSOA_Code", "Year", "Month", TARGET]].copy().head(10)
sample["Predicted"] = np.round(best_ridge_preds[:10], 1)
sample["Error"]     = np.round(np.abs(sample[TARGET] - sample["Predicted"]), 1)
print(sample.to_string(index=False))


--- Sample Predictions vs Actual (first 10 test rows) ---
LSOA_Code  Year  Month  Crime_Count  Predicted  Error
E01000001  2024      1            2        3.9    1.9
E01000001  2024      2            1        3.6    2.6
E01000001  2024      3            2        3.1    1.1
E01000001  2024      4            1        2.5    1.5
E01000001  2024      5            2        1.9    0.1
E01000001  2024      7            1        0.5    0.5
E01000001  2024      8            2        2.2    0.2
E01000001  2024      9            1        0.2    0.8
E01000001  2024     10            4        0.4    3.6
E01000001  2024     11            5        1.4    3.6


In [15]:
# =============================================================================
# CRIME HOTSPOT EVALUATION METRICS
# PAI — Prediction Accuracy Index
# RRI — Recapture Rate Index
# PEI — Prediction Efficiency Index
# =============================================================================
def crime_hotspot_metrics(y_true, y_pred, hotspot_percent=10):

    df_eval = pd.DataFrame({
        "actual": y_true,
        "predicted": y_pred
    }).reset_index(drop=True)

    # Select hotspot cells (top X% predicted crime)
    threshold = np.percentile(df_eval["predicted"], 100 - hotspot_percent)
    df_eval["hotspot"] = df_eval["predicted"] >= threshold

    # Crime counts
    crimes_in_hotspots = df_eval.loc[df_eval["hotspot"], "actual"].sum()
    total_crimes = df_eval["actual"].sum()

    # Area proportions
    hotspot_area = df_eval["hotspot"].sum()
    total_area = len(df_eval)

    # Metrics
    RRI = crimes_in_hotspots / total_crimes
    area_ratio = hotspot_area / total_area
    PAI = RRI / area_ratio
    PEI = RRI / area_ratio   # simplified PEI approximation

    print("\nCrime Hotspot Evaluation")
    print("-" * 40)
    print(f"Hotspot area used : top {hotspot_percent}%")
    print(f"Crimes captured   : {crimes_in_hotspots:.0f} / {total_crimes:.0f}")
    print(f"RRI               : {RRI:.4f}")
    print(f"PAI               : {PAI:.4f}")
    print(f"PEI               : {PEI:.4f}")

    return {
        "Hotspot_%": hotspot_percent,
        "RRI": round(RRI,4),
        "PAI": round(PAI,4),
        "PEI": round(PEI,4)
    }

# =============================================================================
# 13. CRIME HOTSPOT EVALUATION
# =============================================================================
print("\n--- Crime Forecasting Indices ---")

hotspot_results = []

for pct in [5, 10, 20]:
    res = crime_hotspot_metrics(y_test.values, pred_mean, pct)
    hotspot_results.append(res)

hotspot_df = pd.DataFrame(hotspot_results)

print("\nHotspot Evaluation Summary")
print(hotspot_df.to_string(index=False))


--- Crime Forecasting Indices ---


NameError: name 'pred_mean' is not defined